# ISS Group 24 — Two-Model Few-Shot System

**Models**:
1. **Localizer** — multi-shot OWLv2 + learned support fusion. Predicts a bounding
   box conditioned on knowing the object is present (positive-only training).
   Headline metric: **mAP@50** + **containment** (how much of the GT bbox
   the prediction encloses).
2. **Siamese** — multi-shot DINOv2-small + cross-attention head. Predicts
   `existence_prob ∈ [0, 1]` (object present in scene yes/no). Trained with
   focal-BCE + variance + decorrelation regularisers; positive:negative = 1:3.
   Headline metric: **AUROC**, with FPR closely watched (lower is better).

The two models are **fully independent** — train, evaluate, save, and run
inference on either alone. Combined cascaded inference (siamese → localizer)
is provided in `inference_combined.py` with a configurable threshold.

**Stages** (every (epoch, fold) checkpoint is resumable):
- Localizer: Phase 0 (zero-shot OWLv2) → L1 (fusion only) → L2 (+heads) → L3 (+LoRA)
- Siamese:   Phase 0 (zero-shot DINOv2 cosine) → S1 (head only) → S2 (+LoRA)

Set `RUN_SMOKE_TEST = True` to run a ~60s end-to-end smoke check before any
training. Adjust `SHARED_KWARGS` to override any hyperparameter.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import shutil
import subprocess
import time
from pathlib import Path

In [ ]:
SHARED_FOLDER_LINK = "https://drive.google.com/drive/folders/19el_ISdRf5WoXarBsUOXjxFY9A4cPGYJ?usp=sharing"
REPO_URL          = "https://github.com/pewpewnor/iss_group_24.git"
REPO_LOCAL_PATH   = "/content/iss_group_24_src"
USE_GOOGLE_COLAB  = False

# When USE_GOOGLE_COLAB is True we copy the staged dataset from Drive into
# the Colab runtime's local SSD for fast random reads. Checkpoints + analysis
# JSONs always go to the Drive folder so they survive runtime resets.
LOCAL_DATA_ROOT = "/content/iss_group_24_data"


In [ ]:
if USE_GOOGLE_COLAB:
    from google.colab import drive

    def mount_drive() -> str:
        drive.mount("/content/drive")
        for candidate in [
            "/content/drive/Shareddrives/iss_group_24",
            "/content/drive/MyDrive/iss_group_24",
        ]:
            if Path(candidate).exists():
                print(f"Drive mounted. Project root: {candidate}")
                return candidate
        raise RuntimeError("iss_group_24 not found on the mounted Drive")

    DRIVE_ROOT = mount_drive()
else:
    DRIVE_ROOT = str(Path(".").resolve())
    print(f"Local mode. Project root: {DRIVE_ROOT}")

In [ ]:
def setup_repo() -> None:
    if Path(REPO_LOCAL_PATH).exists():
        subprocess.run(["git", "-C", REPO_LOCAL_PATH, "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", REPO_LOCAL_PATH, "reset", "--hard", "origin/main"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth=1", REPO_URL, REPO_LOCAL_PATH], check=True)
    if REPO_LOCAL_PATH not in sys.path:
        sys.path.insert(0, REPO_LOCAL_PATH)


def install_deps() -> None:
    subprocess.run(["pip", "install", "-q",
                    "transformers>=4.40", "accelerate>=0.30", "peft>=0.10",
                    "huggingface_hub>=0.23", "matplotlib>=3.7"], check=True)


if USE_GOOGLE_COLAB:
    setup_repo()
    install_deps()

In [ ]:
# ── Path layout ──────────────────────────────────────────────────────────
# DRIVE_ROOT  : project root on Google Drive (or local repo when not on Colab).
# DATA_ROOT   : where the staged dataset lives at READ time. On Colab this
#               points at the local SSD copy after the next cell mirrors it
#               from Drive (fast random reads). Off Colab it's the Drive copy.
# OUT_ROOT    : where checkpoints are written (always on Drive — survives
#               Colab runtime resets).
# ANALYSIS_ROOT : where per-(epoch, fold) JSON + plots are written (always on Drive).
DRIVE_DATA_ROOT     = f"{DRIVE_ROOT}/dataset/aggregated"
DRIVE_MANIFEST_PATH = f"{DRIVE_DATA_ROOT}/manifest.json"

OUT_ROOT      = f"{DRIVE_ROOT}/checkpoints"
ANALYSIS_ROOT = f"{DRIVE_ROOT}/analysis"

# These two get re-pointed in the next cell when USE_GOOGLE_COLAB is True.
DATA_ROOT = DRIVE_DATA_ROOT
MANIFEST  = DRIVE_MANIFEST_PATH

os.chdir(DRIVE_ROOT)
print(f"Working directory: {os.getcwd()}")
print(f"DRIVE_ROOT     = {DRIVE_ROOT}")
print(f"DATA_ROOT      = {DATA_ROOT}")
print(f"OUT_ROOT       = {OUT_ROOT}")
print(f"ANALYSIS_ROOT  = {ANALYSIS_ROOT}")


---
## Step 0 — Build dataset manifest (writes to Drive)

Stages HOTS + InsDet → `{DRIVE_ROOT}/dataset/aggregated/{train,test}` and vizwiz_novel → `phase0`.
**Idempotent**: re-runs only when staging is missing or invalid. Use
`aggregator.main(force=True)` to rebuild.

Note: this writes to Drive even on Colab — Drive is the source of truth for
the staged dataset and we mirror it locally in the next cell for speed.


In [ ]:
import aggregator
aggregator.main()

---
## Step 0b — (Colab only) Mirror dataset to local runtime SSD

Reads `{DRIVE_ROOT}/dataset/aggregated/` and copies it to `LOCAL_DATA_ROOT`
on the Colab runtime, with progress printed. Drive random-read latency is
~50-100 ms per file; copying once and reading from local SSD takes the
per-file latency down to <1 ms, easily 50× faster training.

After the copy, `DATA_ROOT` and `MANIFEST` are re-pointed to the local copy.
**`OUT_ROOT` and `ANALYSIS_ROOT` continue to point to Drive** so checkpoints
and analysis JSONs persist across runtime resets.

If anything goes wrong (out of disk, permission, etc.) we fall back to reading
directly from Drive.


In [ ]:
def mirror_to_local(src: str, dst: str, *, progress_every: float = 2.0) -> bool:
    """Copy a directory tree from src → dst with periodic progress prints.

    Returns True on success, False otherwise. Falls back to no-op when
    src and dst resolve to the same path.

    Skips files that already exist at dst with the same size + mtime, so
    re-running is cheap.
    """
    src_p = Path(src).resolve()
    dst_p = Path(dst).resolve()
    if src_p == dst_p:
        print(f"  src == dst ({src_p}); nothing to do.")
        return True
    if not src_p.exists():
        print(f"  ✗ source not found: {src_p}")
        return False

    # First pass: enumerate.
    files: list[tuple[Path, Path]] = []
    total_bytes = 0
    for p in src_p.rglob("*"):
        if p.is_file():
            rel = p.relative_to(src_p)
            target = dst_p / rel
            files.append((p, target))
            try:
                total_bytes += p.stat().st_size
            except OSError:
                pass
    n_files = len(files)
    print(f"  found {n_files} files ({total_bytes / 1024 / 1024:.1f} MB) under {src_p}")

    dst_p.mkdir(parents=True, exist_ok=True)
    n_copied = 0
    n_skipped = 0
    bytes_copied = 0
    t_start = time.time()
    t_last = t_start
    for s, d in files:
        try:
            d.parent.mkdir(parents=True, exist_ok=True)
            if d.exists():
                try:
                    s_st = s.stat()
                    d_st = d.stat()
                    if s_st.st_size == d_st.st_size and abs(s_st.st_mtime - d_st.st_mtime) < 1:
                        n_skipped += 1
                        n_copied += 1
                        bytes_copied += s_st.st_size
                        continue
                except OSError:
                    pass
            shutil.copy2(str(s), str(d))
            try:
                bytes_copied += s.stat().st_size
            except OSError:
                pass
            n_copied += 1
        except Exception as e:                                                # noqa: BLE001
            print(f"  ✗ failed to copy {s} → {d}: {e}")
            return False
        now = time.time()
        if now - t_last >= progress_every or n_copied == n_files:
            elapsed = now - t_start
            mb_done = bytes_copied / 1024 / 1024
            mb_total = total_bytes / 1024 / 1024
            rate = mb_done / max(elapsed, 1e-6)
            pct = 100.0 * n_copied / max(n_files, 1)
            eta = (mb_total - mb_done) / max(rate, 1e-6)
            print(
                f"  [{n_copied}/{n_files} = {pct:5.1f}%]  "
                f"{mb_done:7.1f} / {mb_total:7.1f} MB  "
                f"({rate:6.1f} MB/s)  elapsed={elapsed:5.1f}s  eta={eta:5.1f}s",
                flush=True,
            )
            t_last = now
    print(f"  ✓ copied {n_copied} files ({n_skipped} unchanged) in {time.time() - t_start:.1f}s")
    return True


if USE_GOOGLE_COLAB:
    print(f"Mirroring {DRIVE_DATA_ROOT} → {LOCAL_DATA_ROOT} …")
    ok = mirror_to_local(DRIVE_DATA_ROOT, LOCAL_DATA_ROOT)
    if ok:
        DATA_ROOT = LOCAL_DATA_ROOT
        MANIFEST  = f"{LOCAL_DATA_ROOT}/manifest.json"
        print(f"  ✓ DATA_ROOT switched to local: {DATA_ROOT}")
        print(f"  ✓ MANIFEST  switched to local: {MANIFEST}")
    else:
        print("  ⚠ mirror failed; falling back to Drive paths.")
        DATA_ROOT = DRIVE_DATA_ROOT
        MANIFEST  = DRIVE_MANIFEST_PATH
else:
    print("Not on Colab — DATA_ROOT stays at the project's local copy.")

print()
print(f"DATA_ROOT      = {DATA_ROOT}")
print(f"MANIFEST       = {MANIFEST}")
print(f"OUT_ROOT       = {OUT_ROOT}    (always Drive on Colab)")
print(f"ANALYSIS_ROOT  = {ANALYSIS_ROOT}    (always Drive on Colab)")


---
## Step 1 — Smoke test (~60s)

End-to-end check: aggregator validate → localizer L1/L2/L3 → siamese S1/S2 →
combined inference → plots → checkpoint roundtrip. Smoke artifacts go to
`{OUT_ROOT}/_smoke/` and `{ANALYSIS_ROOT}/_smoke/` and are wiped at the end.

Set `RUN_SMOKE_TEST = False` to skip. Recommended on first run.


In [ ]:
RUN_SMOKE_TEST = True

if RUN_SMOKE_TEST:
    from shared.smoke import smoke_test
    res = smoke_test(seconds_budget=60.0, manifest=MANIFEST, data_root=DATA_ROOT)
    print(f"smoke status={res['status']}  wall_clock={res['wall_clock_seconds']:.1f}s")

---
## Step 2 — Imports + shared kwargs

Every parameter in `SHARED_KWARGS` is overridable per-stage. The localizer and
siamese pull defaults from their own `DEFAULT_CFG`; entries below are the most
common knobs you might want to tune.

If a training cell is interrupted (Ctrl-C / kernel-interrupt), the trainer
catches the interrupt and frees CUDA VRAM before re-raising — no kernel
restart needed.


In [ ]:
import torch
from localizer.train import (
    train_phase0    as loc_train_phase0,
    train_stage_L1  as loc_train_L1,
    train_stage_L2  as loc_train_L2,
    train_stage_L3  as loc_train_L3,
    evaluate_phase0 as loc_evaluate_phase0,
    evaluate_run    as loc_evaluate_run,
)
from siamese.train import (
    train_phase0    as sia_train_phase0,
    train_stage_S1  as sia_train_S1,
    train_stage_S2  as sia_train_S2,
    evaluate_run    as sia_evaluate_run,
)
from inference_combined import run_combined, sweep_threshold
from shared.plots import plot_all_from_jsons
from shared.runtime import release_gpu_memory

In [ ]:
# VRAM-aware config tier.
_VRAM = torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else 0
_HAS_BIG_GPU = _VRAM >= 16e9
if _HAS_BIG_GPU:
    LOC_IMG, LOC_BATCH, LOC_ACCUM = 960, 4, 2
    SIA_IMG, SIA_BATCH, SIA_ACCUM = 518, 8, 1
elif _VRAM >= 5.5e9:
    LOC_IMG, LOC_BATCH, LOC_ACCUM = 768, 1, 8
    SIA_IMG, SIA_BATCH, SIA_ACCUM = 518, 4, 2
else:
    LOC_IMG, LOC_BATCH, LOC_ACCUM = 768, 1, 8
    SIA_IMG, SIA_BATCH, SIA_ACCUM = 518, 2, 4
print(f"VRAM={_VRAM/1e9:.1f} GB | localizer img={LOC_IMG} B={LOC_BATCH} accum={LOC_ACCUM} | siamese img={SIA_IMG} B={SIA_BATCH} accum={SIA_ACCUM}")

In [ ]:
# Shared paths/IO/folds — every key is forwardable to both trainers.
SHARED_KWARGS = dict(
    manifest      = MANIFEST,       # local SSD on Colab; Drive elsewhere
    data_root     = DATA_ROOT,      # local SSD on Colab; Drive elsewhere
    out_root      = OUT_ROOT,       # always Drive
    analysis_root = ANALYSIS_ROOT,  # always Drive
    num_workers   = 2,
    use_amp       = True,
    seed          = 42,
    folds         = 3,
    fold_seed     = 42,
    k_min         = 1,
    k_max         = 10,
    keep_last_n   = 0,             # keep every per-(epoch, fold) ckpt
)

# Localizer-specific overrides.
LOC_KWARGS = dict(
    **SHARED_KWARGS,
    img_size         = LOC_IMG,
    batch_size       = LOC_BATCH,
    grad_accum_steps = LOC_ACCUM,
    L1_epochs        = 6,
    L2_epochs        = 12,
    L3_epochs        = 8,
    L1_eps_per_fold  = 200,
    L2_eps_per_fold  = 250,
    L3_eps_per_fold  = 250,
    val_episodes     = 100,
    test_episodes    = 400,
    lr_fusion_L1     = 5e-4,
    lr_fusion_L2     = 1e-4,
    lr_class_L2      = 5e-5,
    lr_box_L2        = 5e-5,
    lr_fusion_L3     = 5e-5,
    lr_class_L3      = 2e-5,
    lr_box_L3        = 2e-5,
    lr_lora_L3       = 1e-4,
    lambda_l1        = 5.0,
    lambda_giou      = 2.0,
    L2_box_warmup_epochs = 2,
    fusion_layers    = 2,
    fusion_heads     = 8,
    fusion_dropout   = 0.1,
    lora_r           = 8,
    lora_alpha       = 16,
    lora_dropout     = 0.1,
    lora_last_n_layers = 4,
    L1_early_stop_patience = 4,
    L2_early_stop_patience = 4,
    L3_early_stop_patience = 4,
)

# Siamese-specific overrides.
SIA_KWARGS = dict(
    **SHARED_KWARGS,
    img_size         = SIA_IMG,
    batch_size       = SIA_BATCH,
    grad_accum_steps = SIA_ACCUM,
    S1_epochs        = 10,
    S2_epochs        = 8,
    S1_eps_per_fold  = 400,
    S2_eps_per_fold  = 400,
    val_episodes     = 200,
    test_episodes    = 400,
    lr_head_S1       = 1e-3,
    lr_head_S2       = 1e-4,
    lr_lora_S2       = 1e-4,
    focal_alpha      = 0.25,         # lower α ⇒ negatives weighted higher ⇒ FP penalty
    focal_gamma      = 2.0,
    variance_target  = 0.5,
    variance_weight  = 0.1,
    decorr_weight    = 0.05,
    neg_prob         = 0.75,         # 1:3 pos:neg
    hard_neg_cache_frac = 0.5,
    cross_attn_heads = 6,
    head_hidden_1    = 256,
    head_hidden_2    = 64,
    head_dropout     = 0.2,
    lora_r           = 8,
    lora_alpha       = 16,
    lora_dropout     = 0.1,
    lora_last_n_layers = 4,
    eval_threshold   = 0.5,
    early_stop_metric = "auroc",     # or "fpr_inv"
    S1_early_stop_patience = 4,
    S2_early_stop_patience = 4,
)

# Localizer

*OWLv2 + multi-shot fusion. Predicts bbox; positive-only training.*

## Localizer Phase 0 — zero-shot OWLv2

In [ ]:
# Zero-shot OWLv2 image-guided detection on phase0 (vizwiz_novel) + test (HOTS+InsDet).
loc_train_phase0(**LOC_KWARGS)

### Evaluate Phase 0 on test split

In [ ]:
loc_evaluate_phase0(**LOC_KWARGS)

## Localizer Stage L1 — fusion transformer only

Freezes OWLv2 entirely; trains only the [CLS]+transformer fusion.

In [ ]:
loc_train_L1(**LOC_KWARGS)

In [ ]:
loc_evaluate_run(checkpoint=f'{OUT_ROOT}/localizer/L1/stage_complete.pt', **LOC_KWARGS)

## Localizer Stage L2 — + class_head + box_head

Unfreezes OWLv2's class_head + box_head + layer_norm. Box head is frozen for the first 2 epochs (configurable via `L2_box_warmup_epochs`).

In [ ]:
loc_train_L2(**LOC_KWARGS)

In [ ]:
loc_evaluate_run(checkpoint=f'{OUT_ROOT}/localizer/L2/stage_complete.pt', **LOC_KWARGS)

## Localizer Stage L3 — + LoRA on last 4 ViT blocks

In [ ]:
loc_train_L3(**LOC_KWARGS)

In [ ]:
loc_evaluate_run(checkpoint=f'{OUT_ROOT}/localizer/L3/stage_complete.pt', **LOC_KWARGS)

---
# Siamese

*DINOv2-small + cross-attention pooling head. Predicts existence_prob; mixed pos:neg = 1:3.*

## Siamese Phase 0 — zero-shot DINOv2 cosine baseline

In [ ]:
sia_train_phase0(**SIA_KWARGS)

## Siamese Stage S1 — cross-attn head only

In [ ]:
sia_train_S1(**SIA_KWARGS)

In [ ]:
sia_evaluate_run(checkpoint=f'{OUT_ROOT}/siamese/S1/stage_complete.pt', **SIA_KWARGS)

## Siamese Stage S2 — + LoRA on last 4 DINOv2 blocks

In [ ]:
sia_train_S2(**SIA_KWARGS)

In [ ]:
sia_evaluate_run(checkpoint=f'{OUT_ROOT}/siamese/S2/stage_complete.pt', **SIA_KWARGS)

---
# Combined cascaded inference

The two models are independent. To deploy: run siamese first, threshold its
`existence_prob`, and (in `"hard"` mode) only invoke the localizer if the
threshold passes. Threshold can be tuned post-hoc with `sweep_threshold`.


## Threshold sweep on the test split

In [ ]:
sweep_threshold(
    siamese_ckpt   = f'{OUT_ROOT}/siamese/S2/stage_complete.pt',
    localizer_ckpt = f'{OUT_ROOT}/localizer/L3/stage_complete.pt',
    manifest       = MANIFEST,
    data_root      = DATA_ROOT,
    test_episodes  = 400,
    thresholds     = (0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70),
    siamese_img_size   = SIA_IMG,
    localizer_img_size = LOC_IMG,
    analysis_root  = ANALYSIS_ROOT,
)

## Single-pair combined inference example

Replace paths with your own files. `existence_threshold_mode` options:
- `"hard"` (default): siamese gates the localizer
- `"soft"`: always run both, return both fields
- `"always_localize"`: always return bbox + existence_prob


In [ ]:
# Example — uncomment and replace paths.
# run_combined(
#     siamese_ckpt   = f'{OUT_ROOT}/siamese/S2/stage_complete.pt',
#     localizer_ckpt = f'{OUT_ROOT}/localizer/L3/stage_complete.pt',
#     support_paths  = ['s1.jpg', 's2.jpg', 's3.jpg', 's4.jpg'],
#     query_path     = 'scene.jpg',
#     existence_threshold      = 0.5,
#     existence_threshold_mode = 'hard',
#     siamese_img_size   = SIA_IMG,
#     localizer_img_size = LOC_IMG,
# )

---
# Plots

In [ ]:
plot_all_from_jsons(ANALYSIS_ROOT)

## Free GPU VRAM

If you ran a training cell and want to release GPU memory without
restarting the kernel (e.g. before opening a different notebook), call
`release_gpu_memory()`.

In [ ]:
release_gpu_memory()